# Clase 2 · Regresión lineal simple en R

Casos prácticos con wage1 y mtcars: de la nube de puntos al ajuste y su interpretación.

Docente: Iván Canales

Ejecuta las celdas en orden con kernel R. En Colab selecciona R en el tipo de entorno; no uses %%R ni rpy2.


## Agenda

- 18:50–19:00: Datos y entorno
- 19:00–19:15: Tres nubes y modelo con error
- 19:15–19:30: Esperanza y promedios por grupo
- 19:30–20:00: MCO y parámetros en R
- 20:00–20:15: Descanso
- 20:15–20:45: Tres regresiones e interpretación
- 20:45–21:05: Residuos, descomposición y R²
- 21:05–21:25: Actividad de comparación y cierre


## Parte 1 · Taller · De los datos a mínimos cuadrados


### Cargar datos y reconocer las unidades

Carga wage1 del libro de Wooldridge. Son 526 personas de Estados Unidos, CPS 1976: wage es salario por hora en USD de 1976; educ, años de educación; exper, años de experiencia potencial; numdep, número de dependientes. Cada fila representa una persona, no un período.


In [ ]:
# Usa la copia local; si no está, carga la base del paquete.
if (file.exists("datos/wage1.csv")) {
  datos <- read.csv("datos/wage1.csv")
} else {
  if (!requireNamespace("wooldridge", quietly = TRUE)) {
    install.packages("wooldridge", repos = "https://cloud.r-project.org/")
  }
  datos <- wooldridge::wage1
}
head(datos)
str(datos)
summary(datos)


**Lectura del resultado:** Observa los nombres y rangos antes de modelar. La experiencia es potencial, no necesariamente experiencia laboral efectivamente acumulada. La columna rownames de la copia CSV es un identificador de fila y no se usa como predictor.


### Nube de puntos: salario y educación

Observa la nube de salario y educación. Antes de mostrar la recta, anticipa la dirección de la tendencia. Luego ejecuta los otros gráficos en R para explorar salario frente a experiencia potencial y número de dependientes.


In [ ]:
# Salario vs Educación
plot(wage ~ educ, data = datos, pch = 19, col = "#005E9080")

# Otros gráficos:
par(mfrow = c(1, 3))
plot(wage ~ exper, data = datos, pch = 19, col = "#005E9080")
plot(wage ~ numdep, data = datos, pch = 19, col = "#005E9080")
par(mfrow = c(1, 1))


**Lectura del resultado:** La nube de salario y educación muestra una tendencia creciente, con dispersión entre personas de un mismo nivel educativo. Recordemos que la recta resume la asociación promedio. La bondad de ajuste es baja: el R² es aproximadamente 16 %, es decir, el modelo explica cerca del 16 % de la variación observada del salario en esta muestra.


### Modelo, error y factores que no observamos

Trabaja con wage = β₀ + β₁ educ + u. Propón factores que podrían integrar u: habilidad, ocupación o condiciones del empleo. Mueve la recta y observa las distancias verticales. El error poblacional u no se observa; los residuos se calculan después de estimar.


In [ ]:
x <- datos$educ
y <- datos$wage
plot(x, y, pch = 19, col = "#005E9080",
     xlab = "Educación (años)", ylab = "Salario por hora (USD de 1976)")
abline(a = 0, b = 0.5, col = "#D6A600", lwd = 3)


**Lectura del resultado:** La recta dibujada es una propuesta inicial, no una estimación. La linealidad es en los parámetros. E[u | x] = 0 es un supuesto sobre la población: esta nube no permite comprobarlo. Una asociación salarial no basta para concluir causalidad.


### Esperanza y esperanza condicional en los datos

Compara el salario promedio de toda la muestra con el promedio dentro de cada nivel educativo. Son promedios muestrales que aproximan esperanzas; no son la función poblacional conocida.


In [ ]:
promedio <- mean(datos$wage)
por_educ <- aggregate(wage ~ educ, data = datos, FUN = mean)
print(promedio)
print(por_educ)
plot(wage ~ educ, data = datos, pch = 19, col = "#005E9040")
points(por_educ$educ, por_educ$wage, pch = 19, col = "#D6A600", cex = 1.4)
abline(h = promedio, lty = 2)


**Lectura del resultado:** Cada punto dorado resume personas con el mismo educ; la línea horizontal resume a todas. Los tamaños de los grupos son diferentes. Bajo E[u | educ] = 0, la regresión poblacional es E[wage | educ] = β₀ + β₁ educ.


### Mínimos cuadrados: estimar con lm()

Ejecuta el código para estimar la recta de salario y educación mediante mínimos cuadrados ordinarios. Los coeficientes los calcula lm() a partir de los datos; en esta lámina no se modifican manualmente.


In [ ]:
modelo_pos <- lm(wage ~ educ, data = datos)
print(summary(modelo_pos))
print(coef(modelo_pos))
print(sum(resid(modelo_pos)^2))
plot(wage ~ educ, data = datos, pch = 19, col = "#005E9060")
abline(modelo_pos, col = "#D6A600", lwd = 3)


**El código, paso a paso**

**modelo_pos <- lm(wage ~ educ, data = datos)**

Estima el salario como función lineal de los años de educación y guarda el ajuste en modelo_pos. La fórmula incluye un intercepto por defecto.

**print(summary(modelo_pos))**

Muestra el resumen del ajuste. Residuals resume los residuos. En Coefficients, Estimate contiene el intercepto (aproximadamente −0.905) y la pendiente (0.541); Std. Error muestra sus errores estándar; t value y Pr(>|t|) corresponden al contraste de que cada coeficiente poblacional sea cero. También aparecen el error estándar residual, R² (aproximadamente 0.1648), R² ajustado y el contraste F global. Estos contrastes requieren los supuestos de la inferencia clásica; no demuestran causalidad.

**print(coef(modelo_pos))**

Extrae e imprime únicamente los dos coeficientes estimados, con sus nombres: (Intercept) y educ. Permite leer la ecuación ŷ = −0.905 + 0.541 educ. La pendiente indica una asociación promedio de 0.541 USD por hora por año adicional de educación.

**print(sum(resid(modelo_pos)^2))**

resid() obtiene las diferencias entre salarios observados y ajustados. ^2 eleva cada residuo al cuadrado y sum() suma esos cuadrados. print() muestra SSR ≈ 5980.68, la suma de cuadrados residual que MCO minimiza. No es un promedio ni R²; está expresada en unidades de salario al cuadrado.

**plot(...) y abline(...)**

plot() dibuja la nube de observaciones. abline(modelo_pos, ...) agrega la recta estimada en dorado; lwd = 3 fija su grosor.


**Lectura del resultado:** MCO minimiza SSR = Σ(yᵢ − ŷᵢ)². El ajuste observado es aproximadamente ŷ = −0.905 + 0.541 educ. Un año adicional de educación se asocia con 0.541 USD/h más, en promedio. El intercepto no debe interpretarse fuera del contexto y soporte de los datos.


### De la fórmula a lm(): regresión muestral

Calcula los parámetros con vectores y compara con lm(). Esta es una estimación en la muestra, no los parámetros poblacionales conocidos.


In [ ]:
x <- datos$educ
y <- datos$wage
b1 <- sum((x - mean(x)) * (y - mean(y))) / sum((x - mean(x))^2)
b0 <- mean(y) - b1 * mean(x)
print(c(b0 = b0, b1 = b1))
print(coef(modelo_pos))
y_ajustado <- b0 + b1 * x
residuos <- y - y_ajustado
print(head(data.frame(y, y_ajustado, residuos)))


**El código, paso a paso**

**x <- datos$educ; y <- datos$wage**

Extraemos dos columnas del dataset y las guardamos como vectores: x contiene los años de educación e y, el salario por hora. Las posiciones coinciden: x[i] e y[i] corresponden a la misma persona. La flecha <- asigna los valores a un nombre.

**b1 <- sum((x - mean(x)) * (y - mean(y))) / sum((x - mean(x))^2)**

Calculamos la pendiente de mínimos cuadrados. mean(x) y mean(y) son los promedios. Al restarlos obtenemos cuánto se aleja cada observación de su promedio. El numerador multiplica esas diferencias persona por persona y las suma: resume cómo varían juntas educación y salario. El denominador suma las diferencias de educación al cuadrado: mide la variación de x y debe ser mayor que cero. Dividir ambos resultados produce b1 ≈ 0.541: por cada año adicional de educación, el salario ajustado aumenta aproximadamente 0.541 USD por hora.

**b0 <- mean(y) - b1 * mean(x)**

Calculamos el intercepto: al salario promedio le restamos la pendiente multiplicada por la educación promedio. Obtenemos b0 ≈ −0.905. Esta elección hace que la recta pase por el punto de los promedios (x̄, ȳ). No significa que exista un salario negativo observado.

**print(c(b0 = b0, b1 = b1))**

c() reúne los dos resultados en un vector con nombres. print() muestra el intercepto y la pendiente calculados con la fórmula. Los nombres b0 y b1 permiten distinguirlos en la salida.

**print(coef(modelo_pos))**

coef() extrae los coeficientes que lm() calculó en la lámina anterior. print() los muestra con los nombres (Intercept) y educ. Deben coincidir con b0 y b1, salvo diferencias mínimas de precisión numérica: estamos aplicando el mismo método. Primero debes ejecutar la celda que crea modelo_pos.

**y_ajustado <- b0 + b1 * x**

Aplicamos la ecuación de la recta a cada persona. R multiplica cada valor de x por b1 y le suma b0; así construye un vector de salarios ajustados. Por ejemplo, con 12 años de educación se obtiene aproximadamente 5.591 USD por hora. Es un valor sobre la recta, no el salario exacto que cada persona debe recibir.

**residuos <- y - y_ajustado**

Restamos, persona por persona, salario observado menos salario ajustado. Un residuo positivo indica que la persona gana más de lo que ajusta el modelo; uno negativo indica que gana menos. Este residuo calculado en la muestra se distingue del error poblacional u, que no observamos.

**print(head(data.frame(y, y_ajustado, residuos)))**

data.frame() coloca los tres vectores como columnas de una tabla; head() selecciona las primeras seis filas y print() las muestra. Compara cada salario real con su ajuste y verifica la resta. En la primera fila: 3.10 − 5.050 ≈ −1.950 USD por hora.


**Lectura del resultado:** Los dos métodos coinciden salvo redondeo numérico. La recta pasa por (x̄, ȳ). No confundas β₁ poblacional con b₁ estimado.


## Parte 2 · Taller · Tres casos, diagnóstico y comparación


### Caso positivo: educación y salario

Consulta summary(modelo_pos) y anova(modelo_pos). Despliega cada salida e interpreta sus componentes con los botones. El modelo explica salario por hora a partir de educación. Predice para 12 y 16 años: ¿es un salario individual garantizado?


In [ ]:
modelo_pos <- lm(wage ~ educ, data = datos)
print(summary(modelo_pos))
print(anova(modelo_pos))
plot(wage ~ educ, data = datos, pch = 19, col = "#005E9060",
     xlab = "Educación (años)", ylab = "Salario por hora (USD de 1976)")
abline(modelo_pos, col = "#D6A600", lwd = 3)
predict(modelo_pos, newdata = data.frame(educ = c(12, 16)))


**Resultados de referencia calculados en R**

**Ver summary(modelo_pos)**

```text
Call:
lm(formula = wage ~ educ, data = datos)

Residuals:
    Min      1Q  Median      3Q     Max
-5.3396 -2.1501 -0.9674  1.1921 16.6085

Coefficients:
            Estimate Std. Error t value Pr(>|t|)
(Intercept) -0.90485    0.68497  -1.321    0.187
educ         0.54136    0.05325  10.167   <2e-16 ***
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

Residual standard error: 3.378 on 524 degrees of freedom
Multiple R-squared:  0.1648,	Adjusted R-squared:  0.1632
F-statistic: 103.4 on 1 and 524 DF,  p-value: < 2.2e-16
```

**Ver anova(modelo_pos)**

```text
Analysis of Variance Table

Response: wage
           Df Sum Sq Mean Sq F value    Pr(>F)
educ        1 1179.7 1179.73  103.36 < 2.2e-16 ***
Residuals 524 5980.7   11.41
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1
```


**El código, paso a paso**

**summary · Estimate: intercepto y pendiente**

Estimate contiene los coeficientes: intercepto ≈ −0.905 y pendiente de educ ≈ 0.541. Un año adicional de educación se asocia con 0.541 USD/h más de salario ajustado. El intercepto es la altura de la recta cuando educ = 0; no es un salario observado negativo.

**summary · Std. Error, t value y Pr(>|t|)**

El error estándar mide la incertidumbre del coeficiente estimado. Para educ es ≈ 0.05325; t = 0.54136/0.05325 ≈ 10.167. Su p-valor ≈ 2.78 × 10⁻²² contrasta H₀: β₁ = 0 bajo los supuestos clásicos. No es la probabilidad de que H₀ sea verdadera ni una medida del tamaño del efecto.

**summary · Residual standard error y R²**

El error estándar residual es ≈ 3.378 USD/h: raíz de SSR/(n−2). R² ≈ 0.1648 indica que la recta explica aproximadamente el 16.48 % de la variación salarial en esta muestra; el R² ajustado es ≈ 0.1632. Una pendiente significativa puede coexistir con baja capacidad explicativa.

**ANOVA · Df: grados de libertad**

La fila educ tiene 1 grado de libertad porque incorporamos una pendiente. Residuals tiene 524 = 526 − 2: estimamos intercepto y pendiente. Los grados de libertad totales son 525 = n − 1.

**ANOVA · Sum Sq: variación explicada y residual**

En la fila educ, Sum Sq ≈ 1179.732 es la suma explicada SSE. En Residuals, Sum Sq ≈ 5980.682 es la suma residual SSR. Ambas suman SST ≈ 7160.414. R² = 1179.732/7160.414 ≈ 0.1648. Seguimos la notación de la cátedra.

**ANOVA · Mean Sq, F value y Pr(>F)**

Mean Sq divide Sum Sq por Df: 1179.732/1 ≈ 1179.732 y 5980.682/524 ≈ 11.4135. F divide ambos cuadrados medios: ≈ 103.36. Pr(>F) es el p-valor del contraste global contra un modelo con sólo intercepto. En esta regresión simple, contrasta β₁ = 0; F = t² y su p-valor coincide con el de la pendiente.

**Predicciones y límites de interpretación**

Para educ = 12 y 16, predict() entrega aproximadamente 5.591 y 7.757 USD/h. Son puntos sobre la recta. Ni la ANOVA ni el p-valor prueban que estudiar cause un cambio salarial; la interpretación inferencial requiere supuestos sobre el error y la muestra.


**Lectura del resultado:** La pendiente es aproximadamente 0.541. Las predicciones son valores medios ajustados, no salarios individuales garantizados. Factores omitidos pueden estar relacionados con educación.


### Tiempo de aceleración: casos negativo y no significativo

Usaremos mtcars, 32 automóviles de modelos 1973–1974, con datos publicados por Motor Trend en 1974. La respuesta es qsec: segundos para recorrer un cuarto de milla (aproximadamente 402 m). hp es potencia bruta en caballos de fuerza; drat es la relación de transmisión del eje trasero, sin unidades. Alterna entre ambos predictores y anticipa el signo. La base viene incluida en R: no necesitas instalar paquetes.


In [ ]:
autos <- datasets::mtcars
head(autos[, c("qsec", "hp", "drat")])

# Caso negativo: potencia y tiempo de aceleración
modelo_neg <- lm(qsec ~ hp, data = autos)
print(summary(modelo_neg))

# Caso no significativo: relación de transmisión y tiempo
modelo_plano <- lm(qsec ~ drat, data = autos)
print(summary(modelo_plano))

par(mfrow = c(1, 2))
plot(qsec ~ hp, data = autos, pch = 19, col = "#005E9080",
     xlab = "Potencia bruta (hp)", ylab = "Tiempo de 1/4 de milla (s)")
abline(modelo_neg, col = "#D6A600", lwd = 3)
plot(qsec ~ drat, data = autos, pch = 19, col = "#005E9080",
     xlab = "Relación de transmisión (sin unidades)",
     ylab = "Tiempo de 1/4 de milla (s)")
abline(modelo_plano, col = "#D6A600", lwd = 3)
par(mfrow = c(1, 1))


**El código, paso a paso**

**Potencia: ¿por qué una pendiente negativa?**

Menos segundos significa completar antes la distancia. Por eso una mayor potencia asociada con un menor tiempo produce una pendiente negativa. Diez hp adicionales se asocian con unos 0.185 segundos menos en el ajuste simple. No estamos comparando el mismo automóvil con y sin una intervención.

**Relación de transmisión: ¿qué significa “neutro”?**

drat describe la relación del eje trasero, no la velocidad medida ni la potencia. Su pendiente es pequeña respecto de su error estándar (0.3048 frente a 0.6076). Con p ≈ 0.62 no hay evidencia suficiente contra β₁ = 0 al 5 %. Esta muestra pequeña no demuestra ausencia de efecto mecánico ni independencia.


**Lectura del resultado:** Con hp, la pendiente es aproximadamente −0.01846 segundos por hp (R² ≈ 0.5016; p ≈ 0.00000577): los autos más potentes tienden a completar la distancia en menos tiempo. Con drat, la pendiente es ≈ 0.3048 segundos por unidad (R² ≈ 0.0083; p ≈ 0.6196): no rechazamos pendiente poblacional cero al 5 %. Este segundo caso es el ejemplo “neutro” en sentido de falta de evidencia lineal, no de pendiente exactamente cero. Son asociaciones entre automóviles distintos; peso, diseño y otras diferencias impiden una conclusión causal.


### Comprobar el caso no significativo con un intervalo

Continúa con mtcars y el modelo qsec ~ drat de la lámina 08. Obtén el intervalo de confianza de la pendiente y compáralo con el p-valor. ¿Incluye el cero? ¿Qué valores positivos y negativos siguen siendo compatibles con estos datos?


In [ ]:
# Ejecuta primero la lámina 08 para crear autos y modelo_plano.
print(summary(modelo_plano))
print(confint(modelo_plano, "drat", level = 0.95))
plot(qsec ~ drat, data = autos, pch = 19, col = "#005E9060",
     xlab = "Relación de transmisión (sin unidades)",
     ylab = "Tiempo de 1/4 de milla (s)")
abline(modelo_plano, col = "#D6A600", lwd = 3)


**Lectura del resultado:** El intervalo del 95 % para la pendiente es aproximadamente [−0.936, 1.546] segundos por unidad de drat. Incluye cero, en concordancia con p ≈ 0.620. La pendiente estimada es positiva, pero no significativa al 5 %. El intervalo también incluye asociaciones de magnitudes relevantes: no podemos concluir equivalencia a cero. Se aplica la inferencia clásica de lm(), condicionada a sus supuestos.


### Residuos y propiedades del ajuste

Activa la animación para dibujar las distancias verticales de los datos a su recta. Comprueba las propiedades algebraicas de MCO con intercepto.


In [ ]:
modelo <- modelo_pos
x <- datos$educ
y <- datos$wage
ajuste <- fitted(modelo)
e <- resid(modelo)
print(c(suma_residuos = sum(e), suma_x_residuos = sum(x * e),
        media_y = mean(y), media_ajustada = mean(ajuste)))
plot(x, y, pch = 19, col = "#005E9060")
segments(x, ajuste, x, y, col = "#D6A60060")
abline(modelo, col = "#D6A600", lwd = 3)


**Lectura del resultado:** Las sumas son prácticamente cero por redondeo de la computadora. Son propiedades del ajuste, no una prueba de E[u | x] = 0 ni del cumplimiento de los supuestos.


### SST, SSE, SSR y R²: ¿cuánto explica la recta?

Compara los tres casos con la misma definición que la cátedra: SST total, SSE explicada y SSR residual. En MCO con intercepto, SST = SSE + SSR.


In [ ]:
modelos <- list(positivo = modelo_pos, negativo = modelo_neg, plano = modelo_plano)
resumen <- do.call(rbind, lapply(modelos, function(m) {
  y <- model.response(model.frame(m))
  SST <- sum((y - mean(y))^2)
  SSE <- sum((fitted(m) - mean(y))^2)
  SSR <- sum(resid(m)^2)
  c(SST = SST, SSE = SSE, SSR = SSR, R2 = 1 - SSR / SST)
}))
print(resumen)
barplot(t(cbind(resumen[, "R2"], 1 - resumen[, "R2"])),
        col = c("#D6A600", "#005E90"),
        legend.text = c("Explicada", "Residual"),
        names.arg = rownames(resumen), ylab = "Fracción de SST")


**Lectura del resultado:** R² = SSE/SST = 1 − SSR/SST. El ajuste salarial explica cerca del 16.48 %; potencia y tiempo, 50.16 %; transmisión y tiempo, 0.83 %. Salario y tiempo son respuestas con unidades y muestras diferentes: compara las fracciones explicadas, no sus SST brutas. El R² no demuestra causalidad.


### Tu turno: comparar y comunicar los tres casos

Completa una tabla con respuesta, predictor, signo, pendiente y R². Redacta una frase en unidades para cada caso y una limitación. Antes de abrir el código, construye la tabla usando coef() y summary().


In [ ]:
comparacion <- data.frame(
  caso = c("Salario y educación", "Tiempo y potencia", "Tiempo y transmisión"),
  pendiente = sapply(modelos, function(m) coef(m)[2]),
  R2 = sapply(modelos, function(m) summary(m)$r.squared),
  p_pendiente = sapply(modelos, function(m) coef(summary(m))[2, 4]))
print(comparacion)
# Escribe tus conclusiones en una celda Markdown del notebook.


**Lectura del resultado:** Revisa dirección, unidades y dispersión alrededor de la recta. El caso no significativo no es una prueba de equivalencia a cero. Diferencia siempre asociación, predicción y causalidad.
